In [2]:
import re
import hashlib
import pandas as pd
from datasets import load_dataset
from langdetect import detect, LangDetectException
from datasketch import MinHash, MinHashLSH
print("All Datasets imported")

/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All Datasets imported


In [7]:
dataset=load_dataset(
    "allenai/c4", #name in huggingface
    "en",        #version
    split="train", #purpose
    streaming=True #dont download whole thing
)
#print(len(list(dataset.take(10))))

'''samples=list(dataset.take(500))
documents=[s["text"] for s in samples]
print(documents[5])

pipeline_stats = {
    "0_original": len(documents)
}'''

BANGALORE CY JUNCTION SBC to GONDIA JUNCTION G train timings, routes, stops, and complete info.
As of now, 1 trains run between from BANGALORE CY JUNCTION (YPR) to GONDIA JUNCTION (G).
The fastest train from BANGALORE CY JUNCTION (YPR) to GONDIA JUNCTION (G) is YPR KRBA WAINGANGA EXP (12251) that departs at 23:40 and arrives to at 21:15. It takes approximately 21:35 hours.


In [8]:


def print_stats(step_name, before, after, removed_docs):
    """Print a clear summary of what each filter removed."""
    removed = before - after
    percent = (removed / before * 100) if before > 0 else 0
    print(f"\n{'='*50}")
    print(f"📊 {step_name}")
    print(f"{'='*50}")
    print(f"  Before : {before} documents")
    print(f"  After  : {after} documents")
    print(f"  Removed: {removed} documents ({percent:.1f}%)")
    if removed_docs:
        print(f"\n  🗑️  Sample removed document:")
        print(f"  {removed_docs[0][:150]}...")
    print(f"{'='*50}")

print("✅ Stats tracker ready!")

✅ Stats tracker ready!


In [9]:
text1="Machine learning is a subset of artificial intelligence. It enables systems to learn from data.Machine learning algorithms improve through experience."
text2="Artificial intelligence includes machine learning and deep learning.Machine learning is a subset of artificial intelligence.Data is important for training models."
text3="Python is widely used in data science.Machine learning algorithms improve through experience.Python libraries help build machine learning models."
text4="Data science involves statistics and programming.Python is widely used in data science.Data is important for training models."
text5="Deep learning is a branch of machine learning.Python libraries help build machine learning models.Artificial intelligence includes machine learning and deep learning."

def get_ngrams(text,n=4):
    words=text.lower().split()
    ngrams=[]
    for i in range(len(words)-n):       
        ngram = " ".join(words[i:i+n])
        ngrams.append(ngram)
    return ngrams
get_ngrams(text2)


['artificial intelligence includes machine',
 'intelligence includes machine learning',
 'includes machine learning and',
 'machine learning and deep',
 'learning and deep learning.machine',
 'and deep learning.machine learning',
 'deep learning.machine learning is',
 'learning.machine learning is a',
 'learning is a subset',
 'is a subset of',
 'a subset of artificial',
 'subset of artificial intelligence.data',
 'of artificial intelligence.data is',
 'artificial intelligence.data is important',
 'intelligence.data is important for',
 'is important for training']

In [30]:
#jaccard

all_sets = [
    set(get_ngrams(text1)),
    set(get_ngrams(text2)),
    set(get_ngrams(text3)),     #jaccard works only for set 
    set(get_ngrams(text4)), 
    set(get_ngrams(text5))
]

for i in range(len(all_sets)):
    for j in range(i + 1, len(all_sets)):
        intersection = all_sets[i].intersection(all_sets[j])
        union = all_sets[i].union(all_sets[j])
        jaccard = len(intersection) / len(union)
        print(i+1,j+1,jaccard)

1 2 0.10344827586206896
1 3 0.034482758620689655
1 4 0.0
1 5 0.0
2 3 0.0
2 4 0.03571428571428571
2 5 0.1
3 4 0.08
3 5 0.06896551724137931
4 5 0.0


In [33]:
all_docs = [
    get_ngrams(text1),
    get_ngrams(text2),
    get_ngrams(text3),
    get_ngrams(text4),
    get_ngrams(text5)
]
all_signatures = []
for doc in all_docs:
    signature = []
    for i in range(128):
        hashes = []
        for shingle in doc:
            hash_value = hash(str(i) + shingle)
            hashes.append(hash_value)
        signature.append(min(hashes))
    all_signatures.append(signature)
len(all_signatures)
#print(all_signatures[0][:10])
#print(all_signatures[1][:10])

5

In [40]:
#Minhash
sig1 = all_signatures[0]
sig2 = all_signatures[1]
sig3 = all_signatures[2]
sig4 = all_signatures[3]
sig5 = all_signatures[4]
signatures=[
    sig1,
    sig2,
    sig3,
    sig4,
    sig5
]
for i in range(len(signatures)):
    for j in range(i+1, len(signatures)):
        matches = 0
        for a, b in zip(signatures[i], signatures[j]):
            if a == b:
                matches += 1
        minhash_similarity = matches / 128
        print(matches)
        print(i+1, j+1, minhash_similarity)
minhash_similarity

15
1 2 0.1171875
4
1 3 0.03125
0
1 4 0.0
0
1 5 0.0
0
2 3 0.0
8
2 4 0.0625
16
2 5 0.125
9
3 4 0.0703125
13
3 5 0.1015625
0
4 5 0.0


0.0

In [42]:
# LSH PARAMETERS
bands = 64
rows_per_band = 2
buckets = {}
# Process every document
for doc_num, signature in enumerate(all_signatures, start=1):
    print(f"\n===== DOC {doc_num} =====")
    # Split signature into bands
    for band in range(bands):
        start = band * rows_per_band
        end = start + rows_per_band
        band_values = signature[start:end]
        print(f"Band {band+1}: {band_values}")
        # Convert list to tuple so it can be used as dictionary key
        bucket_key = (band, tuple(band_values))
        # Create bucket if not exists
        if bucket_key not in buckets:
            buckets[bucket_key] = []
        # Put document into bucket
        buckets[bucket_key].append(doc_num)
print("\n\n========== GROUPS FOUND ==========\n")
# Show only buckets containing more than one document
for bucket_key, docs in buckets.items():

    if len(docs) > 1:

        print(f"Band {bucket_key[0]+1}")
        print(f"Documents: {docs}")
        print("-"*40)


===== DOC 1 =====
Band 1: [-7595214003847391096, -8587847626673050950]
Band 2: [-8833782535911554933, -8122736963302652971]
Band 3: [-7436135823542061618, -9197328355179810185]
Band 4: [-8944191420589075295, -8288098513658912716]
Band 5: [-8571571676184535798, -8826884220864790180]
Band 6: [-8320442624426756532, -9146107592685543790]
Band 7: [-8485668815565086294, -7850204664669679797]
Band 8: [-8850364609891630600, -8515234183943232683]
Band 9: [-9204592526315953305, -8344905984305465280]
Band 10: [-8155036097349634807, -8023584583326928310]
Band 11: [-8469277249074263575, -7520202344709302137]
Band 12: [-6756744601395484309, -8790868985052202427]
Band 13: [-8006258130200335015, -8669372032315540806]
Band 14: [-8356833168608994121, -6794314333833136339]
Band 15: [-5799362173334153629, -8163744956306250606]
Band 16: [-7542643497100514950, -8971377736985182067]
Band 17: [-8816904630970382874, -8286295022151701790]
Band 18: [-7406126493465424086, -8224519983968889578]
Band 19: [-8550747